In [2]:
# ================================================================
# DAY 16 - STUDENT WELLBEING STATISTICAL ANALYSIS & PROBABILITY
# ================================================================
# Complete Jupyter / Google Colab Notebook
# ================================================================


# ================================================================
# 1. IMPORT LIBRARIES
# ================================================================

import pandas as pd
import numpy as np
import os
import glob

from google.colab import files


# ================================================================
# 2. UPLOAD AND LOAD DATASET
# ================================================================

print("=" * 80)
print("STUDENT WELLBEING SURVEY - DAY 16")
print("=" * 80)

uploaded = files.upload()

file_name = next(iter(uploaded))

# Read CSV or Excel
if file_name.lower().endswith(".csv"):
    df = pd.read_csv(file_name)
elif file_name.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(file_name)
else:
    raise ValueError("Please upload a CSV or Excel file.")

print("\nDataset loaded successfully!")
print("File:", file_name)
print("Shape:", df.shape)


# ================================================================
# 3. DATASET INSPECTION
# ================================================================

print("\n" + "=" * 80)
print("1. DATASET INSPECTION")
print("=" * 80)

print("\nFirst 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

print("\nDataset shape:")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())

print("\nTotal missing values:")
print(df.isnull().sum().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nBasic descriptive statistics:")
display(df.describe(include="all"))


# ================================================================
# 4. VERIFY REQUIRED COLUMNS
# ================================================================

required_columns = [
    "Weekly_Study_Hours",
    "Average_Sleep_Hours",
    "Daily_Screen_Time_Hours",
    "Stress_Score",
    "Academic_Readiness_Score",
    "Commute_Time_Minutes",
    "Monthly_Discretionary_Spending",
    "Part_Time_Job",
    "Scholarship",
    "Year_of_Study"
]

# Exercise column:
# The expected dataset column is Exercise_Days_Per_Week.
# If the name differs slightly, automatically search for it.

exercise_column = None

possible_exercise_columns = [
    col for col in df.columns
    if "exercise" in col.lower()
]

if "Exercise_Days_Per_Week" in df.columns:
    exercise_column = "Exercise_Days_Per_Week"
elif len(possible_exercise_columns) == 1:
    exercise_column = possible_exercise_columns[0]

if exercise_column is None:
    raise ValueError(
        "Could not identify the exercise-days column. "
        "Please check the dataset column name."
    )

required_columns.append(exercise_column)

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        "The following required columns are missing:\n"
        + "\n".join(missing_columns)
    )

print("\nAll required columns are available.")
print("Exercise column used:", exercise_column)


# ================================================================
# 5. CONVERT NUMERICAL VARIABLES
# ================================================================

stat_columns = [
    "Weekly_Study_Hours",
    "Average_Sleep_Hours",
    "Daily_Screen_Time_Hours",
    "Stress_Score",
    "Academic_Readiness_Score"
]

outlier_columns = [
    "Weekly_Study_Hours",
    "Daily_Screen_Time_Hours",
    "Commute_Time_Minutes",
    "Monthly_Discretionary_Spending"
]

for column in stat_columns + outlier_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

df[exercise_column] = pd.to_numeric(
    df[exercise_column],
    errors="coerce"
)


# ================================================================
# 6. MEAN, MEDIAN AND MODE
# ================================================================

print("\n" + "=" * 80)
print("2. MEAN, MEDIAN AND MODE")
print("=" * 80)

print("""
FORMULAS

Mean:
    x̄ = Σx / N

Median:
    The middle value after arranging the observations
    in ascending order.

Mode:
    The value that occurs most frequently.
""")

central_tendency = []

for column in stat_columns:

    data = df[column].dropna()

    mean_value = data.mean()
    median_value = data.median()

    modes = data.mode()

    if len(modes) == 0:
        mode_value = "No mode"
    else:
        mode_value = ", ".join(
            [f"{value:.2f}" for value in modes]
        )

    central_tendency.append([
        column,
        mean_value,
        median_value,
        mode_value
    ])

central_tendency_df = pd.DataFrame(
    central_tendency,
    columns=[
        "Variable",
        "Mean",
        "Median",
        "Mode"
    ]
)

display(
    central_tendency_df.style
    .format({
        "Mean": "{:.2f}",
        "Median": "{:.2f}"
    })
)

print("\nINTERPRETATIONS:")

for _, row in central_tendency_df.iterrows():

    variable = row["Variable"]
    mean_value = row["Mean"]
    median_value = row["Median"]

    if mean_value > median_value:
        interpretation = (
            "Mean > median, suggesting possible right-skewness."
        )

    elif mean_value < median_value:
        interpretation = (
            "Mean < median, suggesting possible left-skewness."
        )

    else:
        interpretation = (
            "Mean and median are approximately equal, "
            "suggesting a relatively balanced distribution."
        )

    print(f"{variable}: {interpretation}")


# ================================================================
# 7. RANGE, VARIANCE, STANDARD DEVIATION, Q1, Q3 AND IQR
# ================================================================

print("\n" + "=" * 80)
print("3. MEASURES OF VARIABILITY")
print("=" * 80)

print("""
FORMULAS

Range:
    Range = Maximum - Minimum

Population Variance:
    σ² = Σ(x - μ)² / N

Population Standard Deviation:
    σ = √σ²

First Quartile:
    Q1 = 25th percentile

Third Quartile:
    Q3 = 75th percentile

Interquartile Range:
    IQR = Q3 - Q1
""")

variability_results = []

for column in stat_columns:

    data = df[column].dropna()

    minimum = data.min()
    maximum = data.max()

    range_value = maximum - minimum

    # Population variance and standard deviation
    variance_value = data.var(ddof=0)
    standard_deviation = data.std(ddof=0)

    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)

    IQR = Q3 - Q1

    variability_results.append([
        column,
        minimum,
        maximum,
        range_value,
        variance_value,
        standard_deviation,
        Q1,
        Q3,
        IQR
    ])

variability_df = pd.DataFrame(
    variability_results,
    columns=[
        "Variable",
        "Minimum",
        "Maximum",
        "Range",
        "Variance",
        "Standard Deviation",
        "Q1",
        "Q3",
        "IQR"
    ]
)

display(
    variability_df.style.format({
        "Minimum": "{:.2f}",
        "Maximum": "{:.2f}",
        "Range": "{:.2f}",
        "Variance": "{:.2f}",
        "Standard Deviation": "{:.2f}",
        "Q1": "{:.2f}",
        "Q3": "{:.2f}",
        "IQR": "{:.2f}"
    })
)


# ================================================================
# 8. GREATEST VARIABILITY
# ================================================================

print("\n" + "=" * 80)
print("4. VARIABLE WITH GREATEST VARIABILITY")
print("=" * 80)

greatest_variability = variability_df.loc[
    variability_df["Standard Deviation"].idxmax()
]

print(
    "Greatest variability based on standard deviation:"
)

print(
    "Variable:",
    greatest_variability["Variable"]
)

print(
    "Standard Deviation:",
    round(
        greatest_variability["Standard Deviation"],
        2
    )
)

print(
    "\nINTERPRETATION:"
)

print(
    f"{greatest_variability['Variable']} has the highest "
    f"standard deviation ({greatest_variability['Standard Deviation']:.2f}) "
    "among the five variables. Therefore, it has the greatest "
    "absolute spread around its mean."
)


# ================================================================
# 9. IQR OUTLIER DETECTION
# ================================================================

print("\n" + "=" * 80)
print("5. OUTLIER DETECTION USING THE IQR METHOD")
print("=" * 80)

print("""
FORMULAS

IQR = Q3 - Q1

Lower Fence:
    Q1 - 1.5 × IQR

Upper Fence:
    Q3 + 1.5 × IQR

An observation is an outlier if:

    x < Q1 - 1.5 × IQR

or

    x > Q3 + 1.5 × IQR
""")

outlier_results = []

outlier_values = {}

for column in outlier_columns:

    data = df[column].dropna()

    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)

    IQR = Q3 - Q1

    lower_fence = Q1 - 1.5 * IQR
    upper_fence = Q3 + 1.5 * IQR

    mask = (
        (df[column] < lower_fence) |
        (df[column] > upper_fence)
    )

    outliers = df.loc[
        mask,
        column
    ].dropna()

    outlier_values[column] = outliers.tolist()

    outlier_results.append([
        column,
        Q1,
        Q3,
        IQR,
        lower_fence,
        upper_fence,
        len(outliers)
    ])

    print("\n" + "-" * 70)
    print("Variable:", column)

    print(f"Q1 = {Q1:.2f}")
    print(f"Q3 = {Q3:.2f}")
    print(f"IQR = Q3 - Q1 = {IQR:.2f}")

    print(
        f"Lower Fence = Q1 - 1.5(IQR) = "
        f"{lower_fence:.2f}"
    )

    print(
        f"Upper Fence = Q3 + 1.5(IQR) = "
        f"{upper_fence:.2f}"
    )

    print(
        "Number of outliers:",
        len(outliers)
    )

    if len(outliers) > 0:
        print(
            "Outlier values:",
            [round(x, 2) for x in outliers.tolist()]
        )
    else:
        print("No outliers detected.")

outlier_summary_df = pd.DataFrame(
    outlier_results,
    columns=[
        "Variable",
        "Q1",
        "Q3",
        "IQR",
        "Lower Fence",
        "Upper Fence",
        "Number of Outliers"
    ]
)

print("\nOUTLIER SUMMARY TABLE:")
display(
    outlier_summary_df.style.format({
        "Q1": "{:.2f}",
        "Q3": "{:.2f}",
        "IQR": "{:.2f}",
        "Lower Fence": "{:.2f}",
        "Upper Fence": "{:.2f}"
    })
)


# ================================================================
# 10. BEFORE AND AFTER OUTLIER REMOVAL
# ================================================================

print("\n" + "=" * 80)
print("6. BEFORE VS AFTER OUTLIER REMOVAL")
print("=" * 80)

comparison_variable = "Weekly_Study_Hours"

data = df[comparison_variable].dropna()

Q1 = data.quantile(0.25)
Q3 = data.quantile(0.75)
IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

before_mean = data.mean()
before_median = data.median()

clean_data = data[
    (data >= lower_fence) &
    (data <= upper_fence)
]

after_mean = clean_data.mean()
after_median = clean_data.median()

comparison_df = pd.DataFrame({
    "Measure": ["Mean", "Median"],
    "Before Outlier Removal": [
        before_mean,
        before_median
    ],
    "After Outlier Removal": [
        after_mean,
        after_median
    ]
})

print(
    f"Variable selected: {comparison_variable}"
)

print(
    f"Number of outliers removed: "
    f"{len(data) - len(clean_data)}"
)

display(
    comparison_df.style.format({
        "Before Outlier Removal": "{:.2f}",
        "After Outlier Removal": "{:.2f}"
    })
)

print("\nINTERPRETATION:")

print(
    f"The mean changed from {before_mean:.2f} to "
    f"{after_mean:.2f}, while the median changed from "
    f"{before_median:.2f} to {after_median:.2f}."
)

print(
    "The mean is more sensitive to extreme observations, "
    "whereas the median is generally more resistant to outliers."
)


# ================================================================
# 11. DEFINE PROBABILITY EVENTS
# ================================================================

print("\n" + "=" * 80)
print("7. PROBABILITY EVENTS")
print("=" * 80)

print("""
Event A:
    Student has Part_Time_Job = "Yes"

Event B:
    Student has Stress_Score ≥ 7

Event C:
    Student has Scholarship = "Yes"

Event D:
    Student exercises at least 3 days per week
""")

# Normalize text values
part_time = (
    df["Part_Time_Job"]
    .astype(str)
    .str.strip()
    .str.lower()
)

scholarship = (
    df["Scholarship"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Define events
A = part_time == "yes"

B = df["Stress_Score"] >= 7

C = scholarship == "yes"

D = df[exercise_column] >= 3

N = len(df)

print("Total students:", N)

print("\nEvent counts:")
print("A:", A.sum())
print("B:", B.sum())
print("C:", C.sum())
print("D:", D.sum())


# ================================================================
# 12. BASIC PROBABILITIES
# ================================================================

print("\n" + "=" * 80)
print("8. BASIC PROBABILITIES")
print("=" * 80)

print("""
Formula:

P(E) = Number of outcomes satisfying E / Total number of outcomes
""")

P_A = A.sum() / N
P_B = B.sum() / N
P_C = C.sum() / N
P_D = D.sum() / N

probabilities = pd.DataFrame({
    "Event": ["P(A)", "P(B)", "P(C)", "P(D)"],
    "Count": [
        A.sum(),
        B.sum(),
        C.sum(),
        D.sum()
    ],
    "Probability": [
        P_A,
        P_B,
        P_C,
        P_D
    ],
    "Percentage": [
        P_A * 100,
        P_B * 100,
        P_C * 100,
        P_D * 100
    ]
})

display(
    probabilities.style.format({
        "Probability": "{:.4f}",
        "Percentage": "{:.2f}%"
    })
)

print("\nINTERPRETATIONS:")

print(
    f"P(A) = {P_A:.4f}, meaning {P_A*100:.2f}% "
    "of students have a part-time job."
)

print(
    f"P(B) = {P_B:.4f}, meaning {P_B*100:.2f}% "
    "of students have Stress_Score ≥ 7."
)

print(
    f"P(C) = {P_C:.4f}, meaning {P_C*100:.2f}% "
    "of students have a scholarship."
)

print(
    f"P(D) = {P_D:.4f}, meaning {P_D*100:.2f}% "
    "of students exercise at least 3 days per week."
)


# ================================================================
# 13. P(A OR B) AND P(A AND B)
# ================================================================

print("\n" + "=" * 80)
print("9. UNION AND INTERSECTION PROBABILITIES")
print("=" * 80)

P_A_and_B = (A & B).sum() / N
P_A_or_B = (A | B).sum() / N

print("""
Formula:

P(A ∩ B) = Number of students in both A and B / N

P(A ∪ B) =
P(A) + P(B) - P(A ∩ B)
""")

print(
    f"P(A and B) = {P_A_and_B:.4f} "
    f"= {P_A_and_B*100:.2f}%"
)

print("\nCalculation of P(A or B):")

print(
    f"P(A or B) = {P_A:.4f} + {P_B:.4f} "
    f"- {P_A_and_B:.4f}"
)

print(
    f"P(A or B) = {P_A_or_B:.4f} "
    f"= {P_A_or_B*100:.2f}%"
)

print("\nINTERPRETATION:")

print(
    f"{P_A_and_B*100:.2f}% of students satisfy both "
    "conditions A and B."
)

print(
    f"{P_A_or_B*100:.2f}% satisfy at least one of "
    "the two conditions."
)


# ================================================================
# 14. CONDITIONAL PROBABILITIES
# ================================================================

print("\n" + "=" * 80)
print("10. CONDITIONAL PROBABILITY")
print("=" * 80)

print("""
Formula:

P(A|B) = P(A ∩ B) / P(B)

P(B|A) = P(A ∩ B) / P(A)
""")

if B.sum() == 0:
    raise ValueError("P(B) = 0, so P(A|B) cannot be calculated.")

if A.sum() == 0:
    raise ValueError("P(A) = 0, so P(B|A) cannot be calculated.")

P_A_given_B = (
    (A & B).sum() /
    B.sum()
)

P_B_given_A = (
    (A & B).sum() /
    A.sum()
)

print(
    f"P(A|B) = {P_A_and_B:.4f} / {P_B:.4f}"
)

print(
    f"P(A|B) = {P_A_given_B:.4f} "
    f"= {P_A_given_B*100:.2f}%"
)

print(
    f"\nP(B|A) = {P_A_and_B:.4f} / {P_A:.4f}"
)

print(
    f"P(B|A) = {P_B_given_A:.4f} "
    f"= {P_B_given_A*100:.2f}%"
)

print("\nINTERPRETATION:")

print(
    f"Among students with Stress_Score ≥ 7, "
    f"{P_A_given_B*100:.2f}% have a part-time job."
)

print(
    f"Among students with a part-time job, "
    f"{P_B_given_A*100:.2f}% have Stress_Score ≥ 7."
)


# ================================================================
# 15. MUTUALLY EXCLUSIVE EVENTS
# ================================================================

print("\n" + "=" * 80)
print("11. MUTUALLY EXCLUSIVE EVENTS")
print("=" * 80)

Year1 = df["Year_of_Study"] == 1
Year4 = df["Year_of_Study"] == 4

intersection = Year1 & Year4

P_year1_and_year4 = intersection.sum() / N

print("""
Two events are mutually exclusive when they cannot
occur simultaneously.

Condition:

P(A ∩ B) = 0
""")

print(
    "Year 1 students:",
    Year1.sum()
)

print(
    "Year 4 students:",
    Year4.sum()
)

print(
    "Students who are both Year 1 and Year 4:",
    intersection.sum()
)

print(
    f"P(Year 1 and Year 4) = "
    f"{P_year1_and_year4:.4f}"
)

if intersection.sum() == 0:

    print(
        "\nCONCLUSION: Year 1 and Year 4 are mutually "
        "exclusive events because one student cannot "
        "simultaneously belong to both study years."
    )

else:

    print(
        "\nCONCLUSION: The dataset contains observations "
        "classified as both Year 1 and Year 4, so the "
        "events are not mutually exclusive in the data."
    )


# ================================================================
# 16. INDEPENDENCE OF A AND B
# ================================================================

print("\n" + "=" * 80)
print("12. INDEPENDENCE OF EVENTS A AND B")
print("=" * 80)

print("""
For independent events:

P(A ∩ B) = P(A) × P(B)
""")

product = P_A * P_B

difference = abs(
    P_A_and_B - product
)

print(
    f"P(A and B) = {P_A_and_B:.6f}"
)

print(
    f"P(A) × P(B) = {P_A:.6f} × {P_B:.6f}"
)

print(
    f"P(A) × P(B) = {product:.6f}"
)

print(
    f"Absolute difference = {difference:.6f}"
)

# A small tolerance is used because exact equality
# is uncommon with real survey data.

if np.isclose(
    P_A_and_B,
    product,
    atol=0.01
):

    independence_conclusion = (
        "A and B appear approximately independent."
    )

else:

    independence_conclusion = (
        "A and B do not appear independent."
    )

print("\nCONCLUSION:")
print(independence_conclusion)

if difference > 0.01:

    print(
        "Because the observed joint probability differs "
        "from the product of the individual probabilities, "
        "the dataset suggests an association between "
        "having a part-time job and having Stress_Score ≥ 7."
    )

else:

    print(
        "Because the two probabilities are very close, "
        "the dataset does not provide strong evidence "
        "of dependence between the two events."
    )


# ================================================================
# 17. BAYES' THEOREM
# ================================================================

print("\n" + "=" * 80)
print("13. BAYES' THEOREM")
print("=" * 80)

print("""
Bayes' Theorem:

P(A|B) =
[P(B|A) × P(A)]
------------------------------------------
[P(B|A) × P(A)] +
[P(B|not A) × P(not A)]

where:

P(not A) = 1 - P(A)
""")

not_A = ~A

P_not_A = not_A.sum() / N

if not_A.sum() == 0:
    raise ValueError(
        "There are no students in not-A, so P(B|not A) "
        "cannot be calculated."
    )

P_B_given_not_A = (
    (B & not_A).sum()
    /
    not_A.sum()
)

print(
    f"P(A) = {P_A:.6f}"
)

print(
    f"P(not A) = 1 - P(A) = {P_not_A:.6f}"
)

print(
    f"P(B|A) = {P_B_given_A:.6f}"
)

print(
    f"P(B|not A) = {P_B_given_not_A:.6f}"
)

bayes_numerator = (
    P_B_given_A * P_A
)

bayes_denominator = (
    (P_B_given_A * P_A)
    +
    (P_B_given_not_A * P_not_A)
)

P_A_given_B_bayes = (
    bayes_numerator /
    bayes_denominator
)

print("\nSubstitution:")

print(
    "Numerator = P(B|A) × P(A)"
)

print(
    f"          = {P_B_given_A:.6f} × {P_A:.6f}"
)

print(
    f"          = {bayes_numerator:.6f}"
)

print(
    "\nDenominator = "
    "[P(B|A) × P(A)] + "
    "[P(B|not A) × P(not A)]"
)

print(
    f"            = {bayes_denominator:.6f}"
)

print(
    "\nBayes result:"
)

print(
    f"P(A|B) = {P_A_given_B_bayes:.6f}"
)

print(
    f"P(A|B) = {P_A_given_B_bayes*100:.2f}%"
)


# ================================================================
# 18. VERIFY BAYES AGAINST DIRECT CALCULATION
# ================================================================

print("\n" + "=" * 80)
print("14. BAYES' THEOREM VERIFICATION")
print("=" * 80)

print(
    f"Direct P(A|B) = {P_A_given_B:.10f}"
)

print(
    f"Bayes P(A|B)  = {P_A_given_B_bayes:.10f}"
)

bayes_difference = abs(
    P_A_given_B -
    P_A_given_B_bayes
)

print(
    f"Difference = {bayes_difference:.10f}"
)

if np.isclose(
    P_A_given_B,
    P_A_given_B_bayes,
    atol=1e-10
):

    print(
        "\nVERIFICATION SUCCESSFUL:"
    )

    print(
        "Bayes' theorem agrees with the direct "
        "conditional-probability calculation."
    )

else:

    print(
        "\nThe results are not exactly equal. "
        "Check the calculations and data."
    )


# ================================================================
# 19. NORMAL DISTRIBUTION
# ================================================================

print("\n" + "=" * 80)
print("15. NORMAL DISTRIBUTION - ACADEMIC READINESS SCORE")
print("=" * 80)

academic_scores = (
    df["Academic_Readiness_Score"]
    .dropna()
)

mean_score = academic_scores.mean()

# Population SD, consistent with the variability section
std_score = academic_scores.std(
    ddof=0
)

print("""
Normal Distribution Parameters

Mean:
    μ = Σx / N

Standard deviation:
    σ = √[Σ(x - μ)² / N]

Z-score:
    z = (x - μ) / σ
""")

print(
    f"Mean = {mean_score:.4f}"
)

print(
    f"Standard Deviation = {std_score:.4f}"
)


# ================================================================
# 20. HIGHEST SCORE AND Z-SCORE
# ================================================================

print("\n" + "=" * 80)
print("16. HIGHEST ACADEMIC READINESS SCORE")
print("=" * 80)

highest_score = academic_scores.max()

highest_index = academic_scores.idxmax()

highest_student = df.loc[
    highest_index
]

highest_z = (
    highest_score - mean_score
) / std_score

print(
    "Highest Academic Readiness Score:",
    highest_score
)

# Display Student_ID only if available
if "Student_ID" in df.columns:
    print(
        "Student ID:",
        highest_student["Student_ID"]
    )

print(
    f"Z-score = "
    f"({highest_score:.2f} - {mean_score:.2f}) "
    f"/ {std_score:.2f}"
)

print(
    f"Z-score = {highest_z:.4f}"
)

print("\nINTERPRETATION:")

print(
    f"The highest score is {highest_z:.2f} standard "
    "deviations above the mean."
)

if highest_z >= 3:
    print(
        "This is an extremely high score under the "
        "normal-distribution interpretation."
    )
elif highest_z >= 2:
    print(
        "This is a relatively unusual high score."
    )
else:
    print(
        "This score is within 2 standard deviations "
        "of the mean."
    )


# ================================================================
# 21. LOWEST SCORE AND Z-SCORE
# ================================================================

print("\n" + "=" * 80)
print("17. LOWEST ACADEMIC READINESS SCORE")
print("=" * 80)

lowest_score = academic_scores.min()

lowest_index = academic_scores.idxmin()

lowest_student = df.loc[
    lowest_index
]

lowest_z = (
    lowest_score - mean_score
) / std_score

print(
    "Lowest Academic Readiness Score:",
    lowest_score
)

if "Student_ID" in df.columns:
    print(
        "Student ID:",
        lowest_student["Student_ID"]
    )

print(
    f"Z-score = "
    f"({lowest_score:.2f} - {mean_score:.2f}) "
    f"/ {std_score:.2f}"
)

print(
    f"Z-score = {lowest_z:.4f}"
)

print("\nINTERPRETATION:")

print(
    f"The lowest score is {abs(lowest_z):.2f} standard "
    "deviations below the mean."
)

if lowest_z <= -3:
    print(
        "This is an extremely low score under the "
        "normal-distribution interpretation."
    )
elif lowest_z <= -2:
    print(
        "This is a relatively unusual low score."
    )
else:
    print(
        "This score is within 2 standard deviations "
        "of the mean."
    )


# ================================================================
# 22. 68-95-99.7 EMPIRICAL RULE
# ================================================================

print("\n" + "=" * 80)
print("18. 68-95-99.7 EMPIRICAL RULE")
print("=" * 80)

print("""
For an approximately normal distribution:

Within ±1 standard deviation:
    Approximately 68%

Within ±2 standard deviations:
    Approximately 95%

Within ±3 standard deviations:
    Approximately 99.7%
""")

# Calculate theoretical intervals

lower_1 = mean_score - std_score
upper_1 = mean_score + std_score

lower_2 = mean_score - 2 * std_score
upper_2 = mean_score + 2 * std_score

lower_3 = mean_score - 3 * std_score
upper_3 = mean_score + 3 * std_score

empirical_rule_df = pd.DataFrame({
    "Interval": [
        "Within ±1 SD",
        "Within ±2 SD",
        "Within ±3 SD"
    ],
    "Lower Limit": [
        lower_1,
        lower_2,
        lower_3
    ],
    "Upper Limit": [
        upper_1,
        upper_2,
        upper_3
    ],
    "Expected Percentage": [
        68,
        95,
        99.7
    ]
})

display(
    empirical_rule_df.style.format({
        "Lower Limit": "{:.2f}",
        "Upper Limit": "{:.2f}",
        "Expected Percentage": "{:.1f}%"
    })
)


# ================================================================
# 23. ACTUAL SAMPLE PERCENTAGES
# ================================================================

print("\n" + "=" * 80)
print("19. ACTUAL DATASET PERCENTAGES")
print("=" * 80)

actual_1 = (
    (
        (academic_scores >= lower_1) &
        (academic_scores <= upper_1)
    ).mean() * 100
)

actual_2 = (
    (
        (academic_scores >= lower_2) &
        (academic_scores <= upper_2)
    ).mean() * 100
)

actual_3 = (
    (
        (academic_scores >= lower_3) &
        (academic_scores <= upper_3)
    ).mean() * 100
)

actual_rule_df = pd.DataFrame({
    "Range": [
        "Within ±1 SD",
        "Within ±2 SD",
        "Within ±3 SD"
    ],
    "Actual Percentage": [
        actual_1,
        actual_2,
        actual_3
    ],
    "Theoretical Percentage": [
        68,
        95,
        99.7
    ]
})

display(
    actual_rule_df.style.format({
        "Actual Percentage": "{:.2f}%",
        "Theoretical Percentage": "{:.1f}%"
    })
)

print("\nINTERPRETATION:")

print(
    f"In this dataset, {actual_1:.2f}% of students are "
    "within 1 standard deviation of the mean."
)

print(
    f"{actual_2:.2f}% are within 2 standard deviations "
    "of the mean."
)

print(
    f"{actual_3:.2f}% are within 3 standard deviations "
    "of the mean."
)

print(
    "These values can be compared with the theoretical "
    "68%, 95%, and 99.7% expected under the empirical rule."
)


# ================================================================
# 24. FIVE MEANINGFUL STATISTICAL OBSERVATIONS
# ================================================================

print("\n" + "=" * 80)
print("20. FIVE MEANINGFUL STATISTICAL OBSERVATIONS")
print("=" * 80)


# ---------- Observation 1 ----------

greatest_variable = (
    greatest_variability["Variable"]
)

greatest_sd = (
    greatest_variability["Standard Deviation"]
)

print(
    f"""
1. GREATEST VARIABILITY

{greatest_variable} has the greatest variability among
the five main variables, with a standard deviation of
{greatest_sd:.2f}. A larger standard deviation indicates
greater spread around the mean.
"""
)


# ---------- Observation 2 ----------

max_outlier_variable = max(
    outlier_columns,
    key=lambda x: len(outlier_values[x])
)

max_outlier_count = len(
    outlier_values[max_outlier_variable]
)

print(
    f"""
2. OUTLIERS

{max_outlier_variable} has the largest number of
IQR-detected outliers among the four variables examined,
with {max_outlier_count} outlier(s). This indicates that
this variable contains more observations outside its
1.5 × IQR fences than the other variables.
"""
)


# ---------- Observation 3 ----------

print(
    f"""
3. STUDENT WELLBEING PROBABILITIES

The probability of having a part-time job is
{P_A:.4f} ({P_A*100:.2f}%), while the probability
of having Stress_Score ≥ 7 is {P_B:.4f}
({P_B*100:.2f}%). These probabilities describe the
prevalence of the two conditions in the survey sample.
"""
)


# ---------- Observation 4 ----------

if np.isclose(
    P_A_and_B,
    product,
    atol=0.01
):

    independence_text = (
        f"A and B appear approximately independent because "
        f"P(A and B) = {P_A_and_B:.4f} is close to "
        f"P(A) × P(B) = {product:.4f}."
    )

else:

    independence_text = (
        f"A and B do not appear independent because "
        f"P(A and B) = {P_A_and_B:.4f} differs from "
        f"P(A) × P(B) = {product:.4f}."
    )

print(
    f"""
4. INDEPENDENCE

{independence_text}
This comparison suggests whether having a part-time job
is associated with having a high stress score in this
sample.
"""
)


# ---------- Observation 5 ----------

print(
    f"""
5. ACADEMIC READINESS

The Academic_Readiness_Score has a mean of
{mean_score:.2f} and a standard deviation of
{std_score:.2f}. The highest score has a Z-score of
{highest_z:.2f}, while the lowest score has a Z-score
of {lowest_z:.2f}. Therefore, the highest and lowest
scores lie above and below the average respectively.
"""
)


# ================================================================
# 25. FINAL SUMMARY TABLE
# ================================================================

print("\n" + "=" * 80)
print("21. FINAL RESULTS SUMMARY")
print("=" * 80)

print("\nCENTRAL TENDENCY:")
display(central_tendency_df)

print("\nVARIABILITY:")
display(variability_df)

print("\nOUTLIER SUMMARY:")
display(outlier_summary_df)

print("\nPROBABILITY RESULTS:")
display(probabilities)

print("\nCONDITIONAL PROBABILITIES:")
conditional_df = pd.DataFrame({
    "Probability": [
        "P(A|B)",
        "P(B|A)"
    ],
    "Value": [
        P_A_given_B,
        P_B_given_A
    ],
    "Percentage": [
        P_A_given_B * 100,
        P_B_given_A * 100
    ]
})

display(
    conditional_df.style.format({
        "Value": "{:.4f}",
        "Percentage": "{:.2f}%"
    })
)

print("\nNORMAL DISTRIBUTION:")
normal_df = pd.DataFrame({
    "Measure": [
        "Mean",
        "Standard Deviation",
        "Highest Score",
        "Highest Score Z-score",
        "Lowest Score",
        "Lowest Score Z-score"
    ],
    "Value": [
        mean_score,
        std_score,
        highest_score,
        highest_z,
        lowest_score,
        lowest_z
    ]
})

display(
    normal_df.style.format({
        "Value": "{:.4f}"
    })
)


# ================================================================
# 26. FINAL CONCLUSION
# ================================================================

print("\n" + "=" * 80)
print("FINAL CONCLUSION")
print("=" * 80)

print(
    """
The Student Wellbeing Survey was analyzed using descriptive
statistics, variability measures, the IQR outlier method,
probability, conditional probability, independence,
Bayes' theorem, Z-scores, and the normal-distribution
empirical rule.

The analysis identifies patterns in students' study,
sleep, screen time, stress, academic readiness, exercise,
employment, scholarships, commuting, and discretionary
spending.

All requested calculations and interpretations have been
completed using Python.
"""
)



STUDENT WELLBEING SURVEY - DAY 16


Saving Day16_Student_Wellbeing_Survey (1).csv to Day16_Student_Wellbeing_Survey (1) (1).csv

Dataset loaded successfully!
File: Day16_Student_Wellbeing_Survey (1) (1).csv
Shape: (600, 20)

1. DATASET INSPECTION

First 5 rows:


,Student_ID,Age,Faculty,Year_of_Study,City,Accommodation,Scholarship,Part_Time_Job,Internet_Quality,Preferred_Study_Space,Weekly_Study_Hours,Average_Sleep_Hours,Daily_Screen_Time_Hours,Exercise_Days_Per_Week,Commute_Time_Minutes,Stress_Score,Academic_Readiness_Score,Overall_Satisfaction,Monthly_Discretionary_Spending,Social_Activity_Hours_Per_Week
0,STU0001,21,Data Science,3,Bengaluru,Hostel,No,Yes,Good,Café,15.2,5.7,3.2,3,11.8,5.4,63.1,3.2,5131.0,7.8
1,STU0002,21,Data Science,4,Pune,Hostel,Yes,No,Poor,Library,17.6,7.7,3.5,4,20.3,2.1,74.2,3.8,3508.0,9.2
2,STU0003,22,Life Sciences,2,Chandigarh,Home,Yes,No,Poor,Home,18.9,6.4,2.8,1,15.0,5.2,77.3,4.0,4584.0,7.7
3,STU0004,18,Business,4,Bengaluru,Hostel,Yes,Yes,Average,Café,19.5,6.7,3.0,5,12.1,4.7,67.0,4.0,5456.0,5.3
4,STU0005,20,Business,3,Jammu,Shared Apartment,Yes,No,Good,Home,18.4,7.3,3.0,1,30.2,5.4,65.6,3.2,11732.0,15.9



Last 5 rows:


,Student_ID,Age,Faculty,Year_of_Study,City,Accommodation,Scholarship,Part_Time_Job,Internet_Quality,Preferred_Study_Space,Weekly_Study_Hours,Average_Sleep_Hours,Daily_Screen_Time_Hours,Exercise_Days_Per_Week,Commute_Time_Minutes,Stress_Score,Academic_Readiness_Score,Overall_Satisfaction,Monthly_Discretionary_Spending,Social_Activity_Hours_Per_Week
595,STU0596,20,Engineering,3,Hyderabad,PG,Yes,No,Good,Home,11.7,6.7,5.5,1,21.8,4.5,60.6,4.7,8669.0,11.1
596,STU0597,21,Life Sciences,2,Pune,Shared Apartment,No,No,Average,Department Lab,9.4,7.2,9.9,2,36.1,7.6,75.7,3.8,5169.0,8.0
597,STU0598,18,Data Science,1,Pune,Hostel,No,No,Good,Café,17.3,7.0,4.1,3,11.8,5.0,59.5,4.4,7524.0,14.4
598,STU0599,22,Engineering,3,Hyderabad,Home,No,No,Good,Home,7.9,7.8,1.9,3,44.9,1.8,65.0,4.4,2327.0,7.2
599,STU0600,21,Engineering,1,Jammu,Home,No,No,Average,Library,21.0,7.7,3.0,3,15.0,4.5,85.2,3.4,3236.0,4.7


...


CONDITIONAL PROBABILITIES:


,Probability,Value,Percentage
0,P(A|B),0.6889,68.89%
1,P(B|A),0.2039,20.39%



NORMAL DISTRIBUTION:


,Measure,Value
0,Mean,71.7732
1,Standard Deviation,9.6863
2,Highest Score,98.0000
3,Highest Score Z-score,2.7076
4,Lowest Score,41.9000
5,Lowest Score Z-score,-3.0840



FINAL CONCLUSION

The Student Wellbeing Survey was analyzed using descriptive
statistics, variability measures, the IQR outlier method,
probability, conditional probability, independence,
Bayes' theorem, Z-scores, and the normal-distribution
empirical rule.

The analysis identifies patterns in students' study,
sleep, screen time, stress, academic readiness, exercise,
employment, scholarships, commuting, and discretionary
spending.

All requested calculations and interpretations have been
completed using Python.

